# Embeddings para la revisión del paper 54

**Cómo usarlo**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.
2. Menú **Entorno de ejecución → Ejecutar todo**, y autoriza el acceso a Google Drive cuando lo pida.
3. Deja la pestaña abierta. Todo se guarda en Google Drive a medida que avanza.

**Si se corta o se acaba la GPU gratis:** vuelve a **Ejecutar todo**, aunque sea otro día. Lo que ya
está hecho se salta y sigue donde quedó.

**Qué genera en Google Drive**

- `tabd_embeddings/`: los 4 modelos nuevos, unos 150 MB por modelo. Es lo imprescindible y corre primero.
- `tabd_scale_e5/`: el pool de 1 millón de abstracts para la escalabilidad, unos 4 GB. Tarda varias
  horas y necesita ese espacio libre en Drive. Para omitirlo, desmarca `ESCALA_1M` en la primera celda.

In [ ]:
#@title 1. Preparar el entorno
ESCALA_1M = True  #@param {type:"boolean"}

import base64, io, os, zipfile
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No hay GPU: elige GPU T4 en Entorno de ejecución y vuelve a ejecutar."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tabd_embeddings"
OUT_SCALE = "/content/drive/MyDrive/tabd_scale_e5"
os.makedirs(OUT, exist_ok=True)

BUNDLE = """
...
"""
os.makedirs("/content/tabd", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall("/content/tabd")
os.chdir("/content/tabd")
print("Archivos listos en", os.getcwd())

In [ ]:
#@title 2. Instalar dependencias
!pip install -q datasets sentence-transformers pandas pyarrow

In [ ]:
#@title 3. SciNCL, BERT y E5-base
!python colab_embed.py --models scincl bert e5_base --out "{OUT}"

In [ ]:
#@title 4. SPECTER2 (necesita la librería adapters)
!pip install -q -U adapters
!python colab_embed.py --models specter2 --out "{OUT}"
# adapters puede bajar la versión de transformers: se restaura para la escala
!pip install -q -U transformers sentence-transformers

In [ ]:
#@title 5. Comprobar los modelos
import numpy as np
expected = {"scincl": 768, "bert": 768, "e5_base": 768, "specter2": 768}
counts = {"corpus": 48018, "query": 1000, "title": 1000}
ok = True
for model, dim in expected.items():
    for kind, n in counts.items():
        path = f"{OUT}/{kind}_{model}.npy"
        if not os.path.exists(path):
            print(f"FALTA  {kind}_{model}.npy")
            ok = False
            continue
        shape = np.load(path, mmap_mode="r").shape
        good = shape == (n, dim)
        ok = ok and good
        print(f"{'OK    ' if good else 'MAL   '} {kind}_{model}.npy {shape}")
print("\nTodo correcto." if ok else "\nHay archivos faltantes: copia aquí el error de la celda 3 o 4.")

In [ ]:
#@title 6. Pool de escalabilidad (varias horas; se puede retomar)
if ESCALA_1M:
    !python colab_embed_scale.py --model e5 --n 1000000 --out "{OUT_SCALE}"
else:
    print("Omitido: ESCALA_1M está desmarcado.")